In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from collections import Counter
import ast
import re
pd.set_option('display.max_columns', None)


In [ ]:
data_pendapatan=pd.read_csv('../data/data_pendapatan_cleared_mei_november_2023.csv').drop(columns=['Unnamed: 0'])

# Check Duplicate

In [ ]:
duplicate_rows = data_pendapatan[data_pendapatan.duplicated(keep=False)]
duplicate_rows

In [ ]:
description_counts = data_pendapatan['Description'].value_counts()
description_counts.to_excel('./../data/all_descriptions.xlsx')

In [ ]:
column_to_check = 'Description'
unique_values = data_pendapatan[column_to_check].unique()
unique_values

# RFM
RFM is a method used for analyzing customer value and segmenting the customer base. It is an acronym that stands for:

Recency (R): This metric indicates how recently a customer has made a purchase. A lower recency value means the customer has purchased more recently, indicating higher engagement with the brand.
Frequency (F): This metric signifies how often a customer makes a purchase within a certain period. A higher frequency value indicates a customer who interacts with the business more often, suggesting higher loyalty or satisfaction.
Monetary (M): This metric represents the total amount of money a customer has spent over a certain period. Customers who have a higher monetary value have contributed more to the business, indicating their potential high lifetime value.
Together, these metrics help in understanding a customer's buying behavior and preferences, which is pivotal in personalizing marketing strategies and creating a recommendation system.

In this step, we focus on understanding how recently a customer has made a purchase. This is a crucial aspect of customer segmentation as it helps in identifying the engagement level of customers. Here, I am going to define the following feature:

Days Since Last Purchas: This feature represents the number of days that have passed since the customer's last purchase. A lower value indicates that the customer has purchased recently, implying a higher engagement level with the business, whereas a higher value may indicate a lapse or decreased engagement. By understanding the recency of purchases, businesses can tailor their marketing strategies to re-engage customers who have not made purchases in a while, potentially increasing customer retention and fostering loyalty.

In [ ]:
customer_data = data_pendapatan.groupby('Cust No.')['Tanggal Nota'].max().reset_index()
customer_data

In [ ]:
data_pendapatan['Tanggal Nota'] = pd.to_datetime(data_pendapatan['Tanggal Nota'])
most_recent_date = data_pendapatan['Tanggal Nota'].max()
most_recent_date

In [ ]:
customer_data['Tanggal Nota'] = pd.to_datetime(customer_data['Tanggal Nota'])

In [ ]:
customer_data['Days_Since_Last_Purchase'] = (most_recent_date - customer_data['Tanggal Nota']).dt.days
customer_data

# Frequency

In [ ]:
data_pendapatan[['ID_LEVEL1','LEVEL_1']].value_counts()

In [ ]:
# Menghitung jumlah Cust No. per LEVEL_1
cluster_count = data_pendapatan.groupby('LEVEL_1')['Cust No.'].nunique().reset_index()

# Menghitung persentase
total_cust = cluster_count['Cust No.'].sum()
cluster_count['Percentage'] = (cluster_count['Cust No.'] / total_cust) * 100

# Menyortir berdasarkan LEVEL_1
cluster_count.sort_values(by='LEVEL_1', inplace=True)

# Plotting
plt.figure(figsize=(10, 4))
colors = ['#ff6200', '#ffcaa8', '#ffcaa9', '#ffcaa8', '#ff6200']
sns.barplot(x='Percentage', y='LEVEL_1', data=cluster_count, orient='h', palette=colors)

# Menambahkan label jumlah pelanggan di setiap bar
for index, value in enumerate(cluster_count['Percentage']):
    plt.text(value + 0.5, index, f'{int(cluster_count["Cust No."].iloc[index])} Cust')

plt.title('Distribution of Pendapatan Types Based on Cust No.', fontsize=14)
plt.xticks(ticks=np.arange(0, 50, 5))
plt.xlabel('Percentage (%)')
plt.show()


In [ ]:
# Menghitung jumlah Cust No. per LEVEL_1
cluster_count = data_pendapatan.groupby('LEVEL_1')['Cust No.'].nunique().reset_index()

# Menghitung persentase
total_cust = cluster_count['Cust No.'].sum()
cluster_count['Percentage'] = (cluster_count['Cust No.'] / total_cust) * 100
cluster_count


In [ ]:
aggregate_cust_no = data_pendapatan.groupby(['Cust No.']).agg(
    jumlah_transaksi_pembelian=('Pendapatan in IDR', lambda x: np.sum(x > 0)),
    jumlah_transaksi_retur=('Pendapatan in IDR', lambda x: np.sum(x < 0)),
    median_pendapatan_idr=('Pendapatan in IDR', 'median'),
    min_pendapatan_idr=('Pendapatan in IDR', 'min'),
    max_pendapatan_idr=('Pendapatan in IDR', 'max'),
    rata_rata_pendapatan_idr=('Pendapatan in IDR', 'mean'),
    sum_pendapatan_idr=('Pendapatan in IDR', 'sum'),
)
aggregate_cust_no=aggregate_cust_no.reset_index()
aggregate_cust_no['jumlah_transaksi_keseluruhan']=aggregate_cust_no['jumlah_transaksi_pembelian']+aggregate_cust_no['jumlah_transaksi_retur']
aggregate_cust_no

In [ ]:
aggregate_produk = data_pendapatan.groupby(['Cust No.', 'LEVEL_1']).agg(
    average_transaction=('Pendapatan in IDR', 'mean'),
    freq_transaction=('Pendapatan in IDR', lambda x: np.sum(x > 0)),
    total_transaction=('Pendapatan in IDR', 'sum'),
).reset_index()
aggregate_produk

In [ ]:
product_columns = {}
for product in aggregate_produk['LEVEL_1'].unique():
    freq_col = f'freq_product_{product}'
    avg_col = f'average_product_{product}'    
    total_col = f'sum_product_{product}'    
    product_columns[freq_col] = aggregate_produk[aggregate_produk['LEVEL_1'] == product].groupby('Cust No.')['freq_transaction'].sum()
    product_columns[avg_col] = aggregate_produk[aggregate_produk['LEVEL_1'] == product].groupby('Cust No.')['average_transaction'].mean()
    product_columns[total_col] = aggregate_produk[aggregate_produk['LEVEL_1'] == product].groupby('Cust No.')['total_transaction'].sum()

In [ ]:
product_columns.keys()

In [ ]:
product_columns['freq_product_Pendapatan (Beban) Non Usaha']

In [ ]:
agg=aggregate_cust_no.copy()

In [ ]:
for col, values in product_columns.items():
    values_df = values.reset_index()
    values_df.columns = ['Cust No.', col]
    aggregate_cust_no = aggregate_cust_no.merge(values_df, on='Cust No.', how='left')
aggregate_cust_no


In [ ]:
cust_data=pd.merge(customer_data,aggregate_cust_no,on='Cust No.')
cust_data

In [ ]:
cust_data.to_excel("./../data/customer_data_all_process_v1.xlsx")

# Day Between Transactions

In [ ]:
data_pendapatan['Day_Of_Week'] = data_pendapatan['Tanggal Nota'].dt.dayofweek
data_pendapatan['Week_of_Month'] = data_pendapatan['Tanggal Nota'].dt.day // 7 + 1

days_between_purchases = data_pendapatan.sort_values(by=['Cust No.', 'Tanggal Nota']).groupby('Cust No.')['Tanggal Nota'].apply(lambda x: (x.diff().dropna()).apply(lambda y: y.days))
average_days_between_purchases = days_between_purchases.groupby('Cust No.').mean().reset_index()
average_days_between_purchases.rename(columns={'Tanggal Nota': 'Average_Days_Between_Purchases'}, inplace=True)

# Favorite transaction date and day

In [ ]:
favorite_shopping_day = data_pendapatan.groupby(['Cust No.', 'Day_Of_Week']).size().reset_index(name='Count')
favorite_shopping_day = favorite_shopping_day.loc[favorite_shopping_day.groupby('Cust No.')['Count'].idxmax()][['Cust No.', 'Day_Of_Week']]

In [ ]:
favorite_shopping_week = data_pendapatan.groupby(['Cust No.', 'Week_of_Month']).size().reset_index(name='Count')
favorite_shopping_week = favorite_shopping_week.loc[favorite_shopping_week.groupby('Cust No.')['Count'].idxmax()][['Cust No.', 'Week_of_Month']]

In [ ]:
favorite_shopping_week = data_pendapatan.groupby(['Cust No.', 'Week_of_Month']).size().reset_index(name='Count')
favorite_shopping_week = favorite_shopping_week.loc[favorite_shopping_week.groupby('Cust No.')['Count'].idxmax()][['Cust No.', 'Week_of_Month']]

In [ ]:
customer_data = pd.merge(cust_data, average_days_between_purchases, on='Cust No.')
customer_data = pd.merge(customer_data, favorite_shopping_day, on='Cust No.')
customer_data = pd.merge(customer_data, favorite_shopping_week, on='Cust No.')

In [ ]:
customer_data

In [ ]:
cust_data.to_excel("./../data/customer_data_all_process_v2.xlsx")

# Outliers

In [ ]:
from sklearn.ensemble import IsolationForest


In [ ]:
customer_data_check_outliers=customer_data.drop(columns=['Tanggal Nota'])
customer_data_check_outliers=customer_data_check_outliers.fillna(0)

In [ ]:
print(customer_data_check_outliers.dtypes)


In [ ]:
model = IsolationForest(contamination=0.05, random_state=0)
customer_data_check_outliers['Outlier_Scores'] = model.fit_predict(customer_data_check_outliers.iloc[:, 1:].to_numpy())
customer_data_check_outliers['Is_Outlier'] = [1 if x == -1 else 0 for x in customer_data_check_outliers['Outlier_Scores']]
customer_data_check_outliers.head()

In [ ]:
outlier_percentage = customer_data_check_outliers['Is_Outlier'].value_counts(normalize=True) * 100
plt.figure(figsize=(12, 4))
outlier_percentage.plot(kind='barh', color='#ff6200')

for index, value in enumerate(outlier_percentage):
    plt.text(value, index, f'{value:.2f}%', fontsize=15)

plt.title('Percentage of Inliers and Outliers')
plt.xticks(ticks=np.arange(0, 115, 5))
plt.xlabel('Percentage (%)')
plt.ylabel('Is Outlier')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
outliers_data = customer_data_check_outliers[customer_data_check_outliers['Is_Outlier'] == 1]
customer_data_cleaned = customer_data_check_outliers[customer_data_check_outliers['Is_Outlier'] == 0]
customer_data_cleaned = customer_data_cleaned.drop(columns=['Outlier_Scores', 'Is_Outlier'])
customer_data_cleaned.reset_index(drop=True, inplace=True)

In [ ]:
customer_data_cleaned.to_excel('./../data/customer_data_all_process_v3.xlsx')

In [ ]:
sns.set_style('whitegrid')
corr = customer_data_cleaned.drop(columns=['Cust No.']).corr()
corr

# Trash

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import colors as mcolors
from scipy.stats import linregress
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import KMeans
from tabulate import tabulate
from collections import Counter

%matplotlib inline

In [ ]:
colors = ['#ff6200', '#ffcaa8', 'white', '#ffcaa8', '#ff6200']
my_cmap = LinearSegmentedColormap.from_list('custom_map', colors, N=256)

In [ ]:

mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask, k=1)] = True

plt.figure(figsize=(12, 10))
sns.heatmap(corr, mask=mask, cmap=my_cmap, annot=True, center=0, fmt='.2f', linewidths=2)
plt.title('Correlation Matrix', fontsize=14)
plt.show()

In [ ]:
customer_data_cleaned.columns

In [ ]:
scaler = StandardScaler()
# List of columns that don't need to be scaled
columns_to_exclude = ['Cust No.', 'Day_Of_Week','Week_of_Month']

columns_to_scale = customer_data_cleaned.columns.difference(columns_to_exclude)

customer_data_scaled = customer_data_cleaned.copy()

customer_data_scaled[columns_to_scale] = scaler.fit_transform(customer_data_scaled[columns_to_scale])

customer_data_scaled.head()


In [ ]:
customer_data_scaled.set_index('Cust No.', inplace=True)
pca = PCA().fit(customer_data_scaled)
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance = np.cumsum(explained_variance_ratio)
optimal_k=6
sns.set(rc={'axes.facecolor': '#fcf0dc'}, style='darkgrid')
plt.figure(figsize=(20, 10))
barplot = sns.barplot(x=list(range(1, len(cumulative_explained_variance) + 1)),
                      y=explained_variance_ratio,
                      color='#fcc36d',
                      alpha=0.8)

lineplot, = plt.plot(range(0, len(cumulative_explained_variance)), cumulative_explained_variance,
                     marker='o', linestyle='--', color='#ff6200', linewidth=2)

optimal_k_line = plt.axvline(optimal_k - 1, color='red', linestyle='--', label=f'Optimal k value = {optimal_k}') 

plt.xlabel('Number of Components', fontsize=14)
plt.ylabel('Explained Variance', fontsize=14)
plt.title('Cumulative Variance vs. Number of Components', fontsize=18)
plt.xticks(range(0, len(cumulative_explained_variance)))
plt.legend(handles=[barplot.patches[0], lineplot, optimal_k_line],
           labels=['Explained Variance of Each Component', 'Cumulative Explained Variance', f'Optimal k value = {optimal_k}'],
           loc=(0.62, 0.1),
           frameon=True,
           framealpha=1.0,  
           edgecolor='#ff6200')  

x_offset = -0.3
y_offset = 0.01
for i, (ev_ratio, cum_ev_ratio) in enumerate(zip(explained_variance_ratio, cumulative_explained_variance)):
    plt.text(i, ev_ratio, f"{ev_ratio:.2f}", ha="center", va="bottom", fontsize=10)
    if i > 0:
        plt.text(i + x_offset, cum_ev_ratio + y_offset, f"{cum_ev_ratio:.2f}", ha="center", va="bottom", fontsize=10)

plt.grid(axis='both')   
plt.show()